In [1]:
import numpy as np
import pandas as pd
import xarray as xr

# ============================================================================
# 1. READ IN STATION AND PRODUCT LISTS
# ============================================================================
STNS = np.loadtxt("../DATA/Stations.txt", dtype='U2')
VNS = np.loadtxt("../DATA/Products.txt", dtype='<U10')

# EXCLUDE "TH" PRODUCT FROM ALL ANALYSES
VNS = np.array([v for v in VNS if v != 'TH'])

print(f"Number of stations: {len(STNS)}")
print(f"Number of products: {len(VNS)}")
print(f"Products (TH excluded): {VNS}")
print(f"\nUnits: mm/8days")

# ============================================================================
# 2. LOAD ALL STATION DATA (EXCLUDING TH)
# ============================================================================
print("\nLoading station data (excluding TH)...")

# Initialize with first station, first product
i = 0
STN = STNS[i]
VN = VNS[0]
qwe = xr.open_dataarray("../DATA/"+STN+"/"+VN+"_"+STN+".nc")
DateTime = pd.to_datetime(np.ravel(qwe['Year']*1000 + qwe['DOY']), format='%Y%j')
Values = np.ravel(qwe, order='F').reshape(1, 1, len(DateTime))

STN_DATA = xr.DataArray(data=Values,
                        coords=[[STN], [VN], DateTime],
                        dims=["Station", "Product", "datetime"],
                        name="ET",
                        attrs={'units': 'mm/8days', 
                               'long_name': 'Evapotranspiration'})

# Add remaining products for first station
for j in range(VNS.size-1):
    VN = VNS[j+1]
    qwe = xr.open_dataarray("../DATA/"+STN+"/"+VN+"_"+STN+".nc")
    DateTime = pd.to_datetime(np.ravel(qwe['Year']*1000 + qwe['DOY']), format='%Y%j')
    Values = np.ravel(qwe, order='F').reshape(1, 1, len(DateTime))
    qwe_DATA = xr.DataArray(data=Values,
                            coords=[[STN], [VN], DateTime],
                            dims=["Station", "Product", "datetime"],
                            name="ET",
                            attrs={'units': 'mm/8days', 
                                   'long_name': 'Evapotranspiration'})
    STN_DATA = xr.merge([STN_DATA, qwe_DATA])

# Add all products for remaining stations
for i in range(STNS.size-1):
    STN = STNS[i+1]
    for j in range(VNS.size):
        VN = VNS[j]
        qwe = xr.open_dataarray("../DATA/"+STN+"/"+VN+"_"+STN+".nc")
        DateTime = pd.to_datetime(np.ravel(qwe['Year']*1000 + qwe['DOY']), format='%Y%j')
        Values = np.ravel(qwe, order='F').reshape(1, 1, len(DateTime))
        qwe_DATA = xr.DataArray(data=Values,
                                coords=[[STN], [VN], DateTime],
                                dims=["Station", "Product", "datetime"],
                                name="ET",
                                attrs={'units': 'mm/8days', 
                                       'long_name': 'Evapotranspiration'})
        STN_DATA = xr.merge([STN_DATA, qwe_DATA])

print("Data loaded successfully (TH excluded)!")

# ============================================================================
# 3. CALCULATE BASIC STATISTICS
# ============================================================================
print("\nCalculating errors and basic statistics...")

# Reference data (PM = ground measurements)
ET_ref = STN_DATA['ET'].loc[dict(Product='PM')]

# Errors for each product
STN_ERR = STN_DATA['ET'] - ET_ref
STN_ERR.attrs = {'units': 'mm/8days', 'long_name': 'ET Error'}

# Bias
STN_BIAS = STN_ERR.mean(dim="datetime")
STN_BIAS.attrs = {'units': 'mm/8days', 'long_name': 'ET Bias'}

# RMSE
STN_RMSE = np.sqrt((STN_ERR**2).mean(dim="datetime"))
STN_RMSE.attrs = {'units': 'mm/8days', 'long_name': 'ET Root Mean Square Error'}

# Correlation coefficient
STN_CC = xr.corr(ET_ref, STN_DATA['ET'], dim="datetime")
STN_CC.attrs = {'units': 'dimensionless', 'long_name': 'Correlation Coefficient'}

# ============================================================================
# 4. CALCULATE TAYLOR SKILL SCORES (STS METHOD)
# ============================================================================
print("\nCalculating Taylor Skill Scores...")

# Standard deviation of reference (PM)
std_ref = ET_ref.std(dim="datetime")
std_ref.attrs = {'units': 'mm/8days', 'long_name': 'Standard Deviation of Reference'}

# Standard deviation of each product
std_products = STN_DATA['ET'].std(dim="datetime")
std_products.attrs = {'units': 'mm/8days', 'long_name': 'Standard Deviation'}

# Standard deviation ratio (delta_i in the paper)
delta = std_products / std_ref
delta.attrs = {'units': 'dimensionless', 'long_name': 'Standard Deviation Ratio'}

# Taylor Skill Score formula from paper:
# S_i = 4(1 + R_i)^4 / [(delta_i + 1/delta_i)^2 * (1 + R_max)^4]
# where R_max = 1.0

R_max = 1.0
numerator = 4 * (1 + STN_CC)**4
denominator = (delta + 1/delta)**2 * (1 + R_max)**4

STN_TAYLOR_SKILL = numerator / denominator
STN_TAYLOR_SKILL.attrs = {'units': 'dimensionless', 
                          'long_name': 'Taylor Skill Score',
                          'description': 'Ranges from 0 (worst) to 1 (best)'}

# Handle PM (reference) - set skill to 1.0
STN_TAYLOR_SKILL.loc[dict(Product='PM')] = 1.0

print("\nTaylor Skill Scores by product (averaged across stations):")
for prod in VNS:
    if prod != 'PM':
        skill_mean = STN_TAYLOR_SKILL.loc[dict(Product=prod)].mean().values
        print(f"{prod}: {skill_mean:.4f}")

# ============================================================================
# 5. CALCULATE STS FUSION WEIGHTS
# ============================================================================
print("\nCalculating STS fusion weights...")

# Exclude PM (reference) from fusion
products_to_fuse = [p for p in VNS if p != 'PM']

print(f"Products included in fusion: {products_to_fuse}")

# Get skill scores for products to fuse
skills_to_fuse = STN_TAYLOR_SKILL.loc[dict(Product=products_to_fuse)]

# Calculate weights: W_i = S_i / sum(S_i)
# This is done per station
sum_skills = skills_to_fuse.sum(dim="Product")
STN_WEIGHTS = skills_to_fuse / sum_skills
STN_WEIGHTS.attrs = {'units': 'dimensionless', 
                     'long_name': 'STS Fusion Weights',
                     'description': 'Weights for each product, normalized to sum to 1'}

print("\nWeights by product (averaged across stations):")
for prod in products_to_fuse:
    weight_mean = STN_WEIGHTS.loc[dict(Product=prod)].mean().values
    print(f"{prod}: {weight_mean:.4f} ({weight_mean*100:.2f}%)")

# ============================================================================
# 6. CALCULATE MERGED ET USING STS FUSION
# ============================================================================
print("\nCalculating merged ET (STS)...")

# Get ET data for products to fuse
ET_to_fuse = STN_DATA['ET'].loc[dict(Product=products_to_fuse)]

# Expand weights to match datetime dimension for broadcasting
# weights are [Station, Product], we need [Station, Product, datetime]
weights_expanded = STN_WEIGHTS.expand_dims(dim={'datetime': ET_to_fuse['datetime']})

# Calculate weighted sum: ET_merged = sum(W_i * ET_i)
STN_STS = (ET_to_fuse * weights_expanded).sum(dim="Product")
STN_STS.attrs = {'units': 'mm/8days', 
                 'long_name': 'STS Merged Evapotranspiration',
                 'description': 'Weighted average of ET products using Taylor Skill Scores',
                 'method': 'Simple Taylor Skill (STS) Fusion'}

print("STS merged ET calculated successfully!")

# ============================================================================
# 7. SAVE STS MERGED PRODUCT
# ============================================================================
print("\nSaving STS merged product...")
STN_STS.to_netcdf("../DATA/STN_STS.nc")
print("✓ STS merged product saved as: ../DATA/STN_STS.nc")

# ============================================================================
# 8. LOAD COM MERGED PRODUCT FOR COMPARISON
# ============================================================================
print("\n" + "="*60)
print("LOADING COM PRODUCT FOR COMPARISON")
print("="*60)

try:
    STN_COM = xr.open_dataarray("../DATA/STN_COM.nc")
    print("✓ COM product loaded successfully from: ../DATA/STN_COM.nc")
    print(f"  Shape: {STN_COM.shape}")
    print(f"  Dimensions: {STN_COM.dims}")
except FileNotFoundError:
    print("✗ ERROR: COM product not found at ../DATA/STN_COM.nc")
    print("  Please ensure the file exists before running comparison.")
    raise

# ============================================================================
# 9. HEAD-TO-HEAD COMPARISON: STS vs COM
# ============================================================================
print("\n" + "="*60)
print("HEAD-TO-HEAD COMPARISON: STS vs COM")
print("="*60)

# Calculate statistics for STS
print("\nCalculating STS statistics...")
sts_err = STN_STS - ET_ref
sts_bias = sts_err.mean(dim="datetime")
sts_rmse = np.sqrt((sts_err**2).mean(dim="datetime"))
sts_cc = xr.corr(ET_ref, STN_STS, dim="datetime")

# Calculate Taylor Skill for STS
std_sts = STN_STS.std(dim="datetime")
delta_sts = std_sts / std_ref
numerator_sts = 4 * (1 + sts_cc)**4
denominator_sts = (delta_sts + 1/delta_sts)**2 * (1 + R_max)**4
sts_skill = numerator_sts / denominator_sts

# Calculate statistics for COM
print("Calculating COM statistics...")
com_err = STN_COM - ET_ref
com_bias = com_err.mean(dim="datetime")
com_rmse = np.sqrt((com_err**2).mean(dim="datetime"))
com_cc = xr.corr(ET_ref, STN_COM, dim="datetime")

# Calculate Taylor Skill for COM
std_com = STN_COM.std(dim="datetime")
delta_com = std_com / std_ref
numerator_com = 4 * (1 + com_cc)**4
denominator_com = (delta_com + 1/delta_com)**2 * (1 + R_max)**4
com_skill = numerator_com / denominator_com

# ============================================================================
# 10. PRINT COMPARISON RESULTS
# ============================================================================
print("\n" + "="*60)
print("COMPARISON RESULTS (Averaged Across All Stations)")
print("="*60)

print(f"\n{'Metric':<20} {'STS':<15} {'COM':<15} {'Difference':<15} {'Winner':<10}")
print("-" * 75)

# RMSE Comparison
rmse_sts = sts_rmse.mean().values
rmse_com = com_rmse.mean().values
rmse_diff = rmse_sts - rmse_com
rmse_winner = "STS" if rmse_sts < rmse_com else "COM" if rmse_com < rmse_sts else "TIE"
print(f"{'RMSE (mm/8days)':<20} {rmse_sts:<15.4f} {rmse_com:<15.4f} {rmse_diff:<15.4f} {rmse_winner:<10}")

# Bias Comparison
bias_sts = sts_bias.mean().values
bias_com = com_bias.mean().values
bias_diff = bias_sts - bias_com
bias_winner = "STS" if abs(bias_sts) < abs(bias_com) else "COM" if abs(bias_com) < abs(bias_sts) else "TIE"
print(f"{'Bias (mm/8days)':<20} {bias_sts:<15.4f} {bias_com:<15.4f} {bias_diff:<15.4f} {bias_winner:<10}")

# Correlation Comparison
cc_sts = sts_cc.mean().values
cc_com = com_cc.mean().values
cc_diff = cc_sts - cc_com
cc_winner = "STS" if cc_sts > cc_com else "COM" if cc_com > cc_sts else "TIE"
print(f"{'Correlation':<20} {cc_sts:<15.4f} {cc_com:<15.4f} {cc_diff:<15.4f} {cc_winner:<10}")

# Taylor Skill Comparison
skill_sts = sts_skill.mean().values
skill_com = com_skill.mean().values
skill_diff = skill_sts - skill_com
skill_winner = "STS" if skill_sts > skill_com else "COM" if skill_com > skill_sts else "TIE"
print(f"{'Taylor Skill':<20} {skill_sts:<15.4f} {skill_com:<15.4f} {skill_diff:<15.4f} {skill_winner:<10}")

# Mean ET Comparison
mean_sts = STN_STS.mean().values
mean_com = STN_COM.mean().values
mean_ref = ET_ref.mean().values
mean_diff = mean_sts - mean_com
print(f"\n{'Mean ET (mm/8days)':<20} {mean_sts:<15.2f} {mean_com:<15.2f} {mean_diff:<15.2f}")
print(f"{'Reference (PM)':<20} {mean_ref:<15.2f}")

# Std Dev Comparison
std_sts_val = STN_STS.std().values
std_com_val = STN_COM.std().values
std_ref_val = ET_ref.std().values
std_diff = std_sts_val - std_com_val
print(f"{'StdDev (mm/8days)':<20} {std_sts_val:<15.2f} {std_com_val:<15.2f} {std_diff:<15.2f}")
print(f"{'Reference (PM)':<20} {std_ref_val:<15.2f}")

# ============================================================================
# 11. STATION-BY-STATION COMPARISON
# ============================================================================
print("\n" + "="*60)
print("STATION-BY-STATION COMPARISON")
print("="*60)

station_comparison = []
for stn in STNS:
    sts_rmse_stn = sts_rmse.loc[dict(Station=stn)].values
    com_rmse_stn = com_rmse.loc[dict(Station=stn)].values
    sts_cc_stn = sts_cc.loc[dict(Station=stn)].values
    com_cc_stn = com_cc.loc[dict(Station=stn)].values
    sts_skill_stn = sts_skill.loc[dict(Station=stn)].values
    com_skill_stn = com_skill.loc[dict(Station=stn)].values
    
    station_comparison.append({
        'Station': stn,
        'STS_RMSE': sts_rmse_stn,
        'COM_RMSE': com_rmse_stn,
        'RMSE_Diff': sts_rmse_stn - com_rmse_stn,
        'STS_Corr': sts_cc_stn,
        'COM_Corr': com_cc_stn,
        'Corr_Diff': sts_cc_stn - com_cc_stn,
        'STS_Skill': sts_skill_stn,
        'COM_Skill': com_skill_stn,
        'Skill_Diff': sts_skill_stn - com_skill_stn,
        'Winner_RMSE': 'STS' if sts_rmse_stn < com_rmse_stn else 'COM',
        'Winner_Corr': 'STS' if sts_cc_stn > com_cc_stn else 'COM',
        'Winner_Skill': 'STS' if sts_skill_stn > com_skill_stn else 'COM'
    })

station_comp_df = pd.DataFrame(station_comparison)
station_comp_df = station_comp_df.round({
    'STS_RMSE': 4, 'COM_RMSE': 4, 'RMSE_Diff': 4,
    'STS_Corr': 4, 'COM_Corr': 4, 'Corr_Diff': 4,
    'STS_Skill': 4, 'COM_Skill': 4, 'Skill_Diff': 4
})

# Count wins
sts_rmse_wins = (station_comp_df['Winner_RMSE'] == 'STS').sum()
sts_corr_wins = (station_comp_df['Winner_Corr'] == 'STS').sum()
sts_skill_wins = (station_comp_df['Winner_Skill'] == 'STS').sum()
total_stations = len(STNS)

print(f"\nWins by Metric (out of {total_stations} stations):")
print(f"  RMSE:       STS: {sts_rmse_wins:2d}  |  COM: {total_stations - sts_rmse_wins:2d}")
print(f"  Correlation: STS: {sts_corr_wins:2d}  |  COM: {total_stations - sts_corr_wins:2d}")
print(f"  Skill Score: STS: {sts_skill_wins:2d}  |  COM: {total_stations - sts_skill_wins:2d}")

# ============================================================================
# 12. SAVE COMPARISON RESULTS
# ============================================================================
print("\n" + "="*60)
print("SAVING COMPARISON RESULTS")
print("="*60)

# Save station-by-station comparison
station_comp_df.to_csv("../DATA/STS_vs_COM_by_station.csv", index=False)
print("✓ Station comparison saved: ../DATA/STS_vs_COM_by_station.csv")

# Save summary statistics
summary_comparison = pd.DataFrame({
    'Metric': ['RMSE_mm8d', 'Bias_mm8d', 'Correlation', 'Taylor_Skill', 'Mean_ET_mm8d', 'StdDev_mm8d'],
    'STS': [rmse_sts, bias_sts, cc_sts, skill_sts, mean_sts, std_sts_val],
    'COM': [rmse_com, bias_com, cc_com, skill_com, mean_com, std_com_val],
    'Difference': [rmse_diff, bias_diff, cc_diff, skill_diff, mean_diff, std_diff],
    'Winner': [rmse_winner, bias_winner, cc_winner, skill_winner, 'N/A', 'N/A']
})
summary_comparison.to_csv("../DATA/STS_vs_COM_summary.csv", index=False)
print("✓ Summary comparison saved: ../DATA/STS_vs_COM_summary.csv")

# Save COM statistics as NetCDF for consistency
com_bias.to_netcdf("../DATA/COM_BIAS.nc")
com_rmse.to_netcdf("../DATA/COM_RMSE.nc")
com_cc.to_netcdf("../DATA/COM_CC.nc")
com_skill.to_netcdf("../DATA/COM_SKILL.nc")
print("✓ COM statistics saved as NetCDF files")

# Save STS statistics
sts_bias.to_netcdf("../DATA/STS_BIAS.nc")
sts_rmse.to_netcdf("../DATA/STS_RMSE.nc")
sts_cc.to_netcdf("../DATA/STS_CC.nc")
sts_skill.to_netcdf("../DATA/STS_SKILL.nc")
print("✓ STS statistics saved as NetCDF files")

# ============================================================================
# 13. VISUALIZE IMPROVEMENT
# ============================================================================
print("\n" + "="*60)
print("PERFORMANCE IMPROVEMENT ANALYSIS")
print("="*60)

# Calculate percentage improvements
rmse_improvement = ((rmse_com - rmse_sts) / rmse_com) * 100
bias_improvement = ((abs(bias_com) - abs(bias_sts)) / abs(bias_com)) * 100 if bias_com != 0 else 0
corr_improvement = ((cc_sts - cc_com) / (1 - cc_com)) * 100 if cc_com < 1 else 0
skill_improvement = ((skill_sts - skill_com) / skill_com) * 100 if skill_com != 0 else 0

print(f"\nPercentage Improvements (STS vs COM):")
print(f"  RMSE reduction:        {rmse_improvement:>7.2f}%")
print(f"  Bias reduction:        {bias_improvement:>7.2f}%")
print(f"  Correlation gain:      {corr_improvement:>7.2f}%")
print(f"  Skill score gain:      {skill_improvement:>7.2f}%")

# Overall assessment
print("\n" + "="*60)
print("OVERALL ASSESSMENT")
print("="*60)

metrics_won = sum([
    rmse_sts < rmse_com,
    abs(bias_sts) < abs(bias_com),
    cc_sts > cc_com,
    skill_sts > skill_com
])

print(f"\nSTS wins {metrics_won} out of 4 key metrics")
if metrics_won >= 3:
    print("✓ STS shows SUPERIOR performance overall")
elif metrics_won == 2:
    print("≈ STS shows COMPARABLE performance")
else:
    print("✗ COM shows SUPERIOR performance overall")

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)
print(f"✓ STS product saved as: ../DATA/STN_STS.nc")
print(f"✓ Detailed comparison saved")
print(f"✓ All statistics computed")
print("="*60)

Number of stations: 21
Number of products: 11
Products (TH excluded): ['PM' 'AQUA' 'ERA5' 'GLDAS' 'GLEAM' 'JRA' 'LSA' 'MERRA' 'SSEBop' 'TERRA'
 'WaPOR']

Units: mm/8days

Loading station data (excluding TH)...
Data loaded successfully (TH excluded)!

Calculating errors and basic statistics...

Calculating Taylor Skill Scores...

Taylor Skill Scores by product (averaged across stations):
AQUA: 0.8494
ERA5: 0.7718
GLDAS: 0.8722
GLEAM: 0.8411
JRA: 0.6529
LSA: 0.9508
MERRA: 0.5867
SSEBop: 0.7530
TERRA: 0.8451
WaPOR: 0.7648

Calculating STS fusion weights...
Products included in fusion: [np.str_('AQUA'), np.str_('ERA5'), np.str_('GLDAS'), np.str_('GLEAM'), np.str_('JRA'), np.str_('LSA'), np.str_('MERRA'), np.str_('SSEBop'), np.str_('TERRA'), np.str_('WaPOR')]

Weights by product (averaged across stations):
AQUA: 0.1085 (10.85%)
ERA5: 0.0975 (9.75%)
GLDAS: 0.1106 (11.06%)
GLEAM: 0.1067 (10.67%)
JRA: 0.0798 (7.98%)
LSA: 0.1213 (12.13%)
MERRA: 0.0751 (7.51%)
SSEBop: 0.0959 (9.59%)
TERRA: 0.107